# Hybrid 08 · Extended hybrid tuning: PCA size and `max_features` (follow-up)

**Question:** does extending the PCA size and `max_features` improve the tuned hybrid Random Forest?
**Motivation:** in notebook 13, `max_features = 0.3` (the grid's upper value) was chosen in all 25 folds and PCA 32 (the largest) in 16.

**Derived from:** `notebooks/images/15_extended_hybrid_tuning.ipynb`. **Status: planned follow-up**, the **third** tuning analysis on the same outer folds (after 13 and 14), so it is **not independent confirmation**; each further search on these folds makes a favourable result more likely by chance.
**Design:** as Hybrid 06; only the search space differs. 48 settings: PCA {16, 32, 64, 128} × `max_features` {0.3, 0.5, 0.7} × `min_samples_leaf` {2, 1} × `class_weight` {balanced, None}; `max_depth = 15`, 300 trees. Notebook 13's preprocessing (PCA fitted inside every training fold).

**Runtime and reproduction:** the full run takes ≈12 min on 12 processes. By default the candidate and both references are re-run on one outer fold (≈2 min) and checked against the saved results; then `extended_tuning_fold_scores.csv` (written by notebook 15) is loaded and every comparison recomputed. `RUN_FULL_EXTENDED_TUNING = True` recomputes all 25 folds in memory; nothing is written.

In [1]:
RUN_FULL_EXTENDED_TUNING = False
CHECK_FOLD = (0, 0)

Set up paths, the cohort and the feature matrix.

In [2]:
import os
import sys
import time
import warnings
from pathlib import Path

GRADING_DIR = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "scripts" / "modeling_setup.py").is_file())
MODULE_DIRS = [str(GRADING_DIR / "scripts"), str(GRADING_DIR / "notebooks_cleaned")]
sys.path[:0] = MODULE_DIRS
os.environ["PYTHONPATH"] = os.pathsep.join(MODULE_DIRS + [os.environ.get("PYTHONPATH", "")])

import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from threadpoolctl import threadpool_limits

import cleaned_setup as cs
import modeling_setup as ms

warnings.filterwarnings("ignore", category=UserWarning)
k3, folds, has_ecd = cs.primary_cohort()
X, y, donors = cs.feature_matrix(k3)
P = ms.PROCESSED_DIR
for name in ["extended_tuning_fold_scores.csv", "extended_tuning_selections.csv", "nested_tuning_selections.csv"]:
    print(f"{name}: sha256 {cs.sha256(P / name)}…")

extended_tuning_fold_scores.csv: sha256 717ee0ff9b4fd219…
extended_tuning_selections.csv: sha256 c18973d3678404bc…
nested_tuning_selections.csv: sha256 7ff3077bc130fa81…


## 1. Extended search space (grid order = tie-break order; 0.3 and PCA 16/32 overlap notebook 13's grid)

In [3]:
PCA_GRID = [16, 32, 64, 128]
FOREST_GRID = [{"max_features": f, "min_samples_leaf": leaf, "class_weight": w}
               for f in [0.3, 0.5, 0.7] for leaf in [2, 1] for w in ["balanced", None]]
FOREST_FIXED = dict(n_estimators=300, max_depth=15, criterion="gini", random_state=cs.SEED)
FOREST_JOBS = 1 if RUN_FULL_EXTENDED_TUNING else -1  # forest results do not depend on n_jobs
CANDIDATE, MODELS = "Extended RF hybrid", ["ECD tree", "Tuned RF hybrid", "Extended RF hybrid"]
print(len(PCA_GRID) * len(FOREST_GRID), "settings")

48 settings


Inner tuning and outer refit for the candidate (code from notebook 15), single-threaded BLAS as in its worker processes.

In [4]:
def tune(rows, X, y, donors):
    candidates = [(k, s) for k in PCA_GRID for s in FOREST_GRID]
    scores = np.zeros((3, len(candidates)))
    for i, (tr, va) in enumerate(StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=cs.SEED)
                                 .split(rows, y[rows], groups=donors[rows])):
        tr, va = rows[tr], rows[va]
        assert set(y[va]) == set(cs.CLASSES) and set(donors[tr]).isdisjoint(donors[va])
        for k in PCA_GRID:
            pre = cs.preprocessor(k).fit(X.iloc[tr], y[tr])
            Z_tr, Z_va = pre.transform(X.iloc[tr]), pre.transform(X.iloc[va])
            for j, (kk, setting) in enumerate(candidates):
                if kk == k:
                    forest = RandomForestClassifier(n_jobs=FOREST_JOBS, **FOREST_FIXED, **setting)
                    scores[i, j] = cs.macro_f1(y[va], forest.fit(Z_tr, y[tr]).predict(Z_va))
    mean = scores.mean(axis=0)
    return candidates[int(np.argmax(mean))], mean.max()


def run_candidate(repeat, fold, train, test, X, y, donors, has_ecd):
    with threadpool_limits(1):
        eval_test = test[has_ecd[test]]
        (k, setting), inner = tune(train, X, y, donors)
        model = Pipeline([("features", cs.preprocessor(k)),
                          ("classifier", RandomForestClassifier(n_jobs=FOREST_JOBS, **FOREST_FIXED, **setting))])
        prediction = model.fit(X.iloc[train], y[train]).predict(X.iloc[eval_test])
    return {"repeat": repeat, "fold": fold, "eval_test": eval_test, "prediction": prediction,
            "selection": {"Repeat": repeat, "Fold": fold, "PCA": k, **setting, "Inner macro F1": inner}}

References: the ECD tree and notebook 13's tuned RF hybrid, refitted from its per-fold selections (main process, default threading).

In [5]:
nb13 = pd.read_csv(P / "nested_tuning_selections.csv", dtype={"max_features": str})
nb13_hybrid = nb13[nb13["Arm"] == "Tuned hybrid"]
rf_settings = {(int(r["Repeat"]), int(r["Fold"])): (int(r["PCA"]), {
    "min_samples_leaf": int(r["min_samples_leaf"]), "max_depth": None if pd.isna(r["max_depth"]) else int(r["max_depth"]),
    "max_features": float(r["max_features"]), "class_weight": None if pd.isna(r["class_weight"]) else r["class_weight"]})
    for r in nb13_hybrid.to_dict("records")}


def run_references(repeat, fold, train, test):
    eval_test, train_ecd = test[has_ecd[test]], train[has_ecd[train]]
    k, setting = rf_settings[(repeat, fold)]
    forest = Pipeline([("features", cs.preprocessor(k)),
                       ("classifier", RandomForestClassifier(n_estimators=300, random_state=cs.SEED, n_jobs=-1, **setting))])
    return {"ECD tree": cs.ecd_tree().fit(X.iloc[train_ecd], y[train_ecd]).predict(X.iloc[eval_test]),
            "Tuned RF hybrid": forest.fit(X.iloc[train], y[train]).predict(X.iloc[eval_test])}

## 2. Run and verify against notebook 15

In [6]:
jobs = [(r, f, tr, te) for r in range(ms.N_REPEATS) for f, (tr, te) in enumerate(ms.iter_donor_folds(folds, k3["Tissue_ID"], r))]
N_TRAIN, N_TEST = np.mean([(len(tr), len(te)) for _, _, tr, te in jobs], axis=0)
run_jobs = jobs if RUN_FULL_EXTENDED_TUNING else [j for j in jobs if j[:2] == CHECK_FOLD]

start = time.time()
outputs = Parallel(n_jobs=min(12, os.cpu_count(), len(run_jobs)), backend="loky")(
    delayed(run_candidate)(r, f, tr, te, X, y, donors, has_ecd) for r, f, tr, te in run_jobs)
rows = []
for out, (r, f, tr, te) in zip(outputs, run_jobs):
    for m, pred in {**run_references(r, f, tr, te), CANDIDATE: out["prediction"]}.items():
        rows.append({"Model": m, "Repeat": r, "Fold": f, "N": len(out["eval_test"]),
                     **{k: fn(y[out["eval_test"]], pred) for k, fn in cs.METRICS_WITH_KAPPA.items()}})
live_scores, live_selections = pd.DataFrame(rows), pd.DataFrame([o["selection"] for o in outputs])
print(f"{len(run_jobs)} outer fold(s) in {(time.time() - start) / 60:.1f} min")

saved_scores = pd.read_csv(P / "extended_tuning_fold_scores.csv")
saved_selections = pd.read_csv(P / "extended_tuning_selections.csv")
key = ["Model", "Repeat", "Fold"]
a = live_scores.set_index(key)[list(cs.METRICS_WITH_KAPPA)].sort_index()
print(f"Live vs notebook 15: max absolute metric difference {(a - saved_scores.set_index(key).loc[a.index, list(cs.METRICS_WITH_KAPPA)]).abs().to_numpy().max():.1e}")
cols = ["PCA", "max_features", "min_samples_leaf", "class_weight"]
live_sel = live_selections.set_index(["Repeat", "Fold"])[cols].astype(object).fillna("None").astype(str)
saved_sel = saved_selections.set_index(["Repeat", "Fold"]).loc[live_sel.index, cols].astype(object).fillna("None").astype(str)
print("Selected settings equal notebook 15:", live_sel.equals(saved_sel))

fold_scores = live_scores if RUN_FULL_EXTENDED_TUNING else saved_scores
selections = live_selections if RUN_FULL_EXTENDED_TUNING else saved_selections
print("Source of the 25-fold results:", "recomputed in this session" if RUN_FULL_EXTENDED_TUNING else f"extended_tuning_fold_scores.csv (verified on fold {CHECK_FOLD})")

1 outer fold(s) in 1.4 min
Live vs notebook 15: max absolute metric difference 0.0e+00
Selected settings equal notebook 15: True
Source of the 25-fold results: extended_tuning_fold_scores.csv (verified on fold (0, 0))


## 3. Results on the 470 K4 tissues (fold-level mean ± SD over 25 folds)

In [7]:
cs.summarise(fold_scores, MODELS, cs.METRICS_WITH_KAPPA)

,Macro F1,Balanced accuracy,Accuracy,Recall B,Kappa (quadratic)
ECD tree,0.796 ± 0.038,0.768 ± 0.043,0.804 ± 0.031,0.790 ± 0.117,0.771 ± 0.056
Tuned RF hybrid,0.793 ± 0.047,0.775 ± 0.050,0.797 ± 0.040,0.789 ± 0.130,0.764 ± 0.066
Extended RF hybrid,0.802 ± 0.035,0.781 ± 0.041,0.803 ± 0.029,0.793 ± 0.115,0.770 ± 0.052


F1 (primary, one test, no multiplicity correction): extended − notebook 13's tuned hybrid. F2 (secondary): extended − ECD tree. Checked against `extended_tuning_results.csv`.

In [8]:
results = cs.paired_comparisons(fold_scores, {"F1": (CANDIDATE, "Tuned RF hybrid"), "F2": (CANDIDATE, "ECD tree")},
                                cs.METRICS_WITH_KAPPA, N_TRAIN, N_TEST)
results["Verdict"] = [cs.verdict(d, p) if m == "Macro F1" else "" for d, p, m in zip(results["Difference"], results["p"], results["Metric"])]
saved_results = pd.read_csv(P / "extended_tuning_results.csv")
check = results.merge(saved_results, on=["ID", "Metric"], suffixes=("", " saved"))
print("Max difference from notebook 15's comparisons:",
      f"{max((check[c] - check[c + ' saved']).abs().max() for c in ['Difference', 'CI low', 'CI high', 'p']):.1e}")
results[results["Metric"] == "Macro F1"][["ID", "Comparison", "Model", "Baseline", "Difference", "CI low", "CI high", "p", "Folds improved", "Verdict"]].round(4)

Max difference from notebook 15's comparisons: 1.1e-16


,ID,Comparison,Model,Baseline,Difference,CI low,CI high,p,Folds improved,Verdict
0,F1,Extended RF hybrid − Tuned RF hybrid,0.8018,0.7934,0.0085,-0.0160,0.0330,0.4810,13/25,Inconclusive
5,F2,Extended RF hybrid − ECD tree,0.8018,0.7958,0.0060,-0.0187,0.0307,0.6194,14/25,Inconclusive


Secondary metrics for F1 (unadjusted; all non-significant).

In [9]:
results[(results["ID"] == "F1") & (results["Metric"] != "Macro F1")][["Metric", "Difference", "CI low", "CI high", "p"]].round(4)

,Metric,Difference,CI low,CI high,p
1,Balanced accuracy,0.0055,-0.0142,0.0252,0.5712
2,Accuracy,0.0058,-0.0148,0.0263,0.5681
3,Recall B,0.0040,-0.0326,0.0407,0.8214
4,Kappa (quadratic),0.0060,-0.0209,0.0329,0.6494


Per-grade precision/recall and kappa, pooled over repeats (from the saved out-of-fold predictions).

In [10]:
oof = pd.read_csv(P / "extended_tuning_oof_predictions.csv")
table = cs.per_grade(oof["Actual"], {m: oof[m] for m in MODELS}).T.unstack(level=1).reindex(MODELS)
table["Kappa (quadratic)"] = [cs.kappa(oof["Actual"], oof[m]) for m in MODELS]
table.round(3)

A+                A                B         \
                   Precision Recall Precision Recall Precision Recall   
ECD tree               0.748  0.988     0.884  0.528     1.000  0.791   
Tuned RF hybrid        0.788  0.906     0.761  0.635     0.936  0.791   
Extended RF hybrid     0.786  0.913     0.773  0.639     0.964  0.794   

                   Kappa (quadratic)  
                                      
ECD tree                       0.772  
Tuned RF hybrid                0.769  
Extended RF hybrid             0.774

## 4. Where did the selections land? (counting `None` correctly; notebook 15's printed count omitted it)

In [11]:
for column in ["PCA", "max_features", "min_samples_leaf", "class_weight"]:
    counts = selections[column].astype(object).fillna("None").astype(str).value_counts()
    print(f"{column}: " + ", ".join(f"{v} in {n}" for v, n in counts.items()))
print(f"PCA 128 chosen in {(selections['PCA'] == 128).sum()}/25 folds; max_features 0.7 (new upper edge) in {(selections['max_features'] == 0.7).sum()}/25")
print(f"Mean inner macro F1: {selections['Inner macro F1'].mean():.3f} (notebook 13 hybrid: {nb13_hybrid['Inner macro F1'].mean():.3f})")

PCA: 32 in 14, 16 in 6, 64 in 5
max_features: 0.7 in 13, 0.5 in 8, 0.3 in 4
min_samples_leaf: 2 in 15, 1 in 10
class_weight: None in 13, balanced in 12
PCA 128 chosen in 0/25 folds; max_features 0.7 (new upper edge) in 13/25
Mean inner macro F1: 0.767 (notebook 13 hybrid: 0.760)


## 5. Application rule (fixed before the run)

Replace the deployed model only if the extended hybrid convincingly outperforms it: difference > 0, p < 0.05 and grade B recall difference ≥ −0.02.

In [12]:
f1 = results[(results["ID"] == "F1") & (results["Metric"] == "Macro F1")].iloc[0]
b_diff = results[(results["ID"] == "F1") & (results["Metric"] == "Recall B")]["Difference"].iloc[0]
convincing = bool(f1["Difference"] > 0 and f1["p"] < 0.05 and b_diff >= -0.02)
print(f"Extended − tuned RF hybrid: {f1['Difference']:+.4f} [{f1['CI low']:+.4f}, {f1['CI high']:+.4f}], p = {f1['p']:.2f}; "
      f"B recall difference {b_diff:+.4f} -> {'replacement recommended' if convincing else 'the deployed tuned RF hybrid is retained'}")

Extended − tuned RF hybrid: +0.0085 [-0.0160, +0.0330], p = 0.48; B recall difference +0.0040 -> the deployed tuned RF hybrid is retained


## Interpretation (follow-up on reused folds)

- The extended hybrid scored macro F1 0.802, accuracy 0.803 and grade B recall 0.793, against 0.793 / 0.797 / 0.789 for the deployed configuration. The difference, **+0.008 [−0.016, +0.033], p = 0.48**, is **inconclusive**: a higher point estimate is not an improvement.
- Against the ECD tree it is +0.006 [−0.019, +0.031], also inconclusive.
- Larger image representations were not preferred (PCA 128 never chosen; PCA 32 most common). `max_features = 0.7`, the new upper edge, was chosen in 13 of 25 folds: a reason for further search on **new tissues**, not evidence of improvement.
- Across notebooks 13–15, three tuning analyses on the same folds all land at ≈0.79–0.80 macro F1 and ≈0.80 accuracy, the level of the two-threshold ECD tree.

**Limitations:** third tuning-related analysis on the same folds; no independent test set; the selection rule was fixed before the run but not committed separately.